In [ ]:
import os
import pandas as pd

MODEL_ID = "google/medgemma-4b-it"
MODEL_ID = os.getenv("MODEL_ID", MODEL_ID)
results_dir = f"results/{MODEL_ID.split('/')[-1]}"

orig_data = pd.read_json(f"{results_dir}/medqa_mcq_with_disease_mentions_and_answers.json", lines=True)
tr_data = pd.read_json(f"{results_dir}/tr_medqa_mcq_with_disease_mentions_and_answers.json", lines=True)

#structural counts from the raw data, independent of any later prediction-parsing filtering
n_total_questions = len(orig_data)
n_total_transformed = len(tr_data)
n_questions_with_transformed = orig_data["index"].isin(tr_data["oid"]).sum()

print(f"total questions in dataset:           {n_total_questions}")
print(f"total questions with >=1 transformed: {n_questions_with_transformed}")
print(f"total transformed questions:          {n_total_transformed}")

In [ ]:
#drop rows where the predicted answer is None/NaN
n_orig_unparsed = orig_data["predicted_answer"].isna().sum()
n_tr_unparsed = tr_data["predicted_answer"].isna().sum()
print(f"dropping {n_orig_unparsed}/{len(orig_data)} unparsed original predictions")
print(f"dropping {n_tr_unparsed}/{len(tr_data)} unparsed transformed predictions")

orig_data = orig_data.dropna(subset=["predicted_answer"])
tr_data = tr_data.dropna(subset=["predicted_answer"])

In [ ]:
#tr_data's "oid" is the original question's row index in orig_data ("index"), so each
#transformed question can be matched back to the original prediction it was derived from.
merged = tr_data.merge(
    orig_data[["index", "predicted_answer", "gold_answer"]],
    left_on="oid", right_on="index", suffixes=("_trans", "_orig"),
)
print(len(merged), "transformed questions matched to an original prediction")

In [ ]:
#restricted to original questions that have at least one transformed counterpart, so this is
#a paired comparison against transformed_accuracy/FR/NFR/PFR rather than two different populations
orig_data_paired = orig_data[orig_data["index"].isin(tr_data["oid"])]
orig_accuracy = (orig_data_paired["predicted_answer"] == orig_data_paired["gold_answer"]).mean()

#for context only -- accuracy over the full original dataset, not just the paired subset
orig_accuracy_full = (orig_data["predicted_answer"] == orig_data["gold_answer"]).mean()

transformed_accuracy = (tr_data["predicted_answer"] == tr_data["gold_answer"]).mean()

flip_rate = (merged["predicted_answer_trans"] != merged["predicted_answer_orig"]).mean()

negative_flip_rate = (
    (merged["predicted_answer_orig"] == merged["gold_answer_trans"])
    & (merged["predicted_answer_trans"] != merged["gold_answer_trans"])
).mean()

positive_flip_rate = (
    (merged["predicted_answer_orig"] != merged["gold_answer_trans"])
    & (merged["predicted_answer_trans"] == merged["gold_answer_trans"])
).mean()

print(f"original accuracy (paired, n={len(orig_data_paired)}): {orig_accuracy:.4f}")
print(f"original accuracy (full dataset, n={len(orig_data)}):  {orig_accuracy_full:.4f}")
print(f"transformed accuracy:      {transformed_accuracy:.4f}")
print(f"flip rate (FR):            {flip_rate:.4f}")
print(f"negative flip rate (NFR):  {negative_flip_rate:.4f}")
print(f"positive flip rate (PFR):  {positive_flip_rate:.4f}")